# Pré-processamento 3 — Arapiraca

Este notebook transforma os dados de ocorrências em uma matriz de série temporal:
1. Carrega o CSV com índice de célula do grid
2. Exploração inicial dos dados (anos disponíveis)
3. Calcula a hora do ano para cada ocorrência
4. Conta ocorrências por célula × hora (todos os tipos juntos), monta uma matriz completa e salva em um único CSV

**Entrada:** `arapiraca_with_grid_index.csv`

**Saída:** `arapiraca_all_crimes_grid.csv` (matriz célula × hora, contagem agregada de todos os tipos selecionados)


## 1. Carregamento dos dados com índice de célula

In [ ]:
# Carrega o CSV de Arapiraca com a coluna 'grid_cell_id' (índice da célula do grid)
# Converte DATA_HORA para datetime

import pandas as pd

df_crimes = pd.read_csv(
    "./output/arapiraca/arapiraca_with_grid_index.csv",
    delimiter=";",
    parse_dates=["DATA_HORA"]
)

## 2. Exploração dos dados

In [2]:
display(df_crimes)
df_crimes.info()
years = sorted(df_crimes['DATA_HORA'].dt.year.unique())
display(years)

,Unnamed: 0,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO,cell
0,0,-36.690147,-9.735983,street_robbery,2022-01-01 03:00:00,Arapiraca,2256
1,1,-36.655517,-9.785501,street_robbery,2022-01-01 20:00:00,Arapiraca,1732
2,2,-36.591520,-9.762967,vehicle_robbery,2022-01-02 09:00:00,Arapiraca,768
3,3,-36.664139,-9.768197,street_robbery,2022-01-02 23:00:00,Arapiraca,1850
4,4,-36.680604,-9.644294,street_robbery,2022-01-03 08:00:00,Arapiraca,2163
...,...,...,...,...,...,...,...
17366,17366,-36.662065,-9.764001,vehicle_robbery,2014-12-21 01:30:00,Arapiraca,1851
17367,17367,-36.666260,-9.753766,street_robbery,2014-12-22 14:30:00,Arapiraca,1910
17368,17368,-36.663400,-9.748675,street_robbery,2014-12-22 22:00:00,Arapiraca,1854
17369,17369,-36.648386,-9.721165,vehicle_robbery,2014-12-22 23:30:00,Arapiraca,1633


<class 'pandas.DataFrame'>
RangeIndex: 17371 entries, 0 to 17370
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Unnamed: 0   17371 non-null  int64         
 1   LONGITUDE    17371 non-null  float64       
 2   LATITUDE     17371 non-null  float64       
 3   ROUBO_GROUP  17371 non-null  str           
 4   DATA_HORA    17371 non-null  datetime64[us]
 5   CIDADE_FATO  17371 non-null  str           
 6   cell         17371 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 950.1 KB


[np.int32(2012),
 np.int32(2013),
 np.int32(2014),
 np.int32(2015),
 np.int32(2016),
 np.int32(2017),
 np.int32(2018),
 np.int32(2019),
 np.int32(2020),
 np.int32(2021),
 np.int32(2022)]

## 4. Cálculo da hora do ano

Para cada ocorrência, calcula quantas horas se passaram desde o início do ano. Isso permite organizar os dados em uma grade temporal uniforme.

In [4]:
# Calcula a hora do ano para cada ocorrência
# Exemplo: 1 de janeiro 00:00 = hora 0, 2 de janeiro 00:00 = hora 24
# Utiliza pandarallel para processamento paralelo

import numpy as np
import datetime
from pandarallel import pandarallel
import os

cpu_count = os.cpu_count() or 1
nb_workers = max(1, min(cpu_count, 24))
pandarallel.initialize(nb_workers=nb_workers, progress_bar=True)


def hour_of_year(dt):
    beginning_of_year = datetime.datetime(
        dt["DATA_HORA"].year, 1, 1, tzinfo=dt["DATA_HORA"].tzinfo
    )
    return pd.Series(
        {
            "hour_year": (dt["DATA_HORA"] - beginning_of_year).total_seconds()
            // 3600
        }
    )


df_crimes
display("Translate date of crime to hour of the year")
df_hour = df_crimes.merge(
    df_crimes.parallel_apply(hour_of_year, axis=1), left_index=True, right_index=True
)
display("Initial shape: ", df_hour.shape)
display("Initial shape: ", df_hour)

INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


'Translate date of crime to hour of the year'

'Initial shape: '

(17371, 8)

'Initial shape: '

,Unnamed: 0,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO,cell,hour_year
0,0,-36.690147,-9.735983,street_robbery,2022-01-01 03:00:00,Arapiraca,2256,3.0
1,1,-36.655517,-9.785501,street_robbery,2022-01-01 20:00:00,Arapiraca,1732,20.0
2,2,-36.591520,-9.762967,vehicle_robbery,2022-01-02 09:00:00,Arapiraca,768,33.0
3,3,-36.664139,-9.768197,street_robbery,2022-01-02 23:00:00,Arapiraca,1850,47.0
4,4,-36.680604,-9.644294,street_robbery,2022-01-03 08:00:00,Arapiraca,2163,56.0
...,...,...,...,...,...,...,...,...
17366,17366,-36.662065,-9.764001,vehicle_robbery,2014-12-21 01:30:00,Arapiraca,1851,8497.0
17367,17367,-36.666260,-9.753766,street_robbery,2014-12-22 14:30:00,Arapiraca,1910,8534.0
17368,17368,-36.663400,-9.748675,street_robbery,2014-12-22 22:00:00,Arapiraca,1854,8542.0
17369,17369,-36.648386,-9.721165,vehicle_robbery,2014-12-22 23:30:00,Arapiraca,1633,8543.0


## 3. Construção da matriz agregada de ocorrências

Para cada ano, conta o número de ocorrências por célula e hora do ano (somando todos os tipos selecionados em pre-processing). Gera uma matriz completa (célula × hora). Todas as células e horas sem ocorrência são preenchidas com NaN. Os anos são concatenados e o resultado é salvo em um único CSV.

In [ ]:
# Conta ocorrências de TODOS os tipos selecionados juntas (sem separar por ROUBO_GROUP)
# Para cada ano, gera matriz célula × hora_do_ano e concatena horizontalmente

# Valor de GRID_SIZE calculado no notebook pre-processing-1
GRID_SIZE = 57

df_all_years_list = []
for y in years:
    dfy = df_hour[df_hour["DATA_HORA"].dt.year == y].copy()

    # Conta ocorrências por célula e hora do ano (somando todos os tipos)
    counts = dfy.groupby(["grid_cell_id", "hour_year"]).size()

    # Linhas = células, colunas = horas do ano
    grid = counts.unstack("hour_year")

    # Total de horas no ano (considera bissexto)
    beginning_of_y1 = datetime.datetime(y, 1, 1)
    beginning_of_y2 = datetime.datetime(y + 1, 1, 1)
    num_hours_total = int((beginning_of_y2 - beginning_of_y1).total_seconds() // 3600)

    # Garante que todas as horas e células estão presentes (NaN onde não há dados)
    grid = grid.reindex(columns=range(num_hours_total))
    grid = grid.reindex(index=range(GRID_SIZE ** 2))

    # Renomeia colunas adicionando sufixo do ano
    df_year = grid.copy()
    df_year.columns = [f"{c}_{y}" for c in df_year.columns.tolist()]
    df_all_years_list.append(df_year)

# Concatena todos os anos lado a lado (linhas = células, colunas = todas as horas de todos os anos)
df_all = pd.concat(df_all_years_list, axis=1)
# Transpõe (linhas = horas, colunas = células) — formato esperado pelo pre-trainning
df_all = df_all.T
print("Shape final (horas × células):", df_all.shape)

df_all.to_csv("./output/arapiraca/arapiraca_all_crimes_grid.csv")
print("Salvo em ./output/arapiraca/arapiraca_all_crimes_grid.csv")
